In [3]:
from sklearn.datasets import load_iris
import pandas as pd 
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

dataset = load_iris()
dataset.keys()

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])

In [2]:
print(dataset.DESCR)

.. _iris_dataset:

Iris plants dataset
--------------------

**Data Set Characteristics:**

:Number of Instances: 150 (50 in each of three classes)
:Number of Attributes: 4 numeric, predictive attributes and the class
:Attribute Information:
    - sepal length in cm
    - sepal width in cm
    - petal length in cm
    - petal width in cm
    - class:
            - Iris-Setosa
            - Iris-Versicolour
            - Iris-Virginica

:Summary Statistics:

============== ==== ==== ======= ===== ====================
                Min  Max   Mean    SD   Class Correlation
============== ==== ==== ======= ===== ====================
sepal length:   4.3  7.9   5.84   0.83    0.7826
sepal width:    2.0  4.4   3.05   0.43   -0.4194
petal length:   1.0  6.9   3.76   1.76    0.9490  (high!)
petal width:    0.1  2.5   1.20   0.76    0.9565  (high!)
============== ==== ==== ======= ===== ====================

:Missing Attribute Values: None
:Class Distribution: 33.3% for each of 3 classes.
:Cr

In [12]:
data = pd.DataFrame(dataset.data, columns=dataset.feature_names)
data['target'] = dataset.target

print(data.head())
print(data.shape)

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  
(150, 5)


In [8]:
# 파이토치 데이터 유틸 사용하기 p108

class IrisDataset(Dataset):
  def __init__(self, train=True):
    dataset = load_iris()
    X_tr, X_ts, y_tr, y_ts = train_test_split(
      dataset.data, dataset.target, test_size=0.3, random_state= 827
    )
    if train:
      self.data = torch.FloatTensor(X_tr)
      self.target = torch.LongTensor(y_tr)
    else:
      self.data = torch.FloatTensor(X_ts)
      self.target = torch.LongTensor(y_ts)

  def __getitem__(self, i):
    return self.data[i], self.target[i]
  
  def __len__(self):
    return len(self.data)

In [11]:
# 3.3 모델 구현 및 학습
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ----------------------------
# 하이퍼파라미터 설정
# ----------------------------
batch_size = 64         # 한 번에 처리할 데이터 수
learning_rate = 1e-3    # 학습률
epochs = 2000           # 총 학습 횟수

# ----------------------------
# 모델 정의 (순차 모델 사용)
# 입력 4차원 → 출력 3차원 (Iris 클래스 수)
# ----------------------------
model = nn.Sequential(
    nn.Linear(4, 128),  # 입력층: 4개 feature → 128개 노드
    nn.ReLU(),          # 비선형 활성화 함수
    nn.Linear(128, 64), # 은닉층: 128 → 64
    nn.ReLU(),
    nn.Linear(64, 3)    # 출력층: 3개 클래스 (softmax는 CrossEntropyLoss에서 자동 적용됨)
)

# ----------------------------
# 데이터셋 및 데이터로더 정의
# IrisDataset은 사용자 정의 클래스라고 가정
# ----------------------------
train_dataset = IrisDataset(train=True)  # 학습용 데이터셋 로드
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)  # 셔플하여 배치 구성

# ----------------------------
# 옵티마이저 정의 (Adam 사용)
# ----------------------------
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)

# ----------------------------
# 손실 함수 정의 (CrossEntropyLoss는 softmax + log-loss)
# ----------------------------
criterion = nn.CrossEntropyLoss()

# ----------------------------
# 학습 루프 시작
# ----------------------------
for epoch in range(epochs):

    # 각 배치마다 모델 학습
    for data, target in train_dataloader:
        optimizer.zero_grad()         # 이전 gradient 초기화
        pred = model(data)            # 모델 예측 수행
        loss = criterion(pred, target)  # 손실 계산
        loss.backward()              # 역전파로 gradient 계산
        optimizer.step()             # 가중치 업데이트

    # 100 epoch마다 현재 손실 출력
    if epoch % 100 == 99:
        print("epoch:", epoch+1, "loss:", loss.item())


epoch: 100 loss: 0.07362639904022217
epoch: 200 loss: 0.030425434932112694
epoch: 300 loss: 0.03564496338367462
epoch: 400 loss: 0.03852378576993942
epoch: 500 loss: 0.10320969671010971
epoch: 600 loss: 0.04004615917801857
epoch: 700 loss: 0.006972684990614653
epoch: 800 loss: 0.03630082681775093
epoch: 900 loss: 0.014710146002471447
epoch: 1000 loss: 0.011239002458751202
epoch: 1100 loss: 0.03734271228313446
epoch: 1200 loss: 0.015614934265613556
epoch: 1300 loss: 0.03802711144089699
epoch: 1400 loss: 0.05343884974718094
epoch: 1500 loss: 0.012751458212733269
epoch: 1600 loss: 0.03766165301203728
epoch: 1700 loss: 0.042871665209531784
epoch: 1800 loss: 0.07104938477277756
epoch: 1900 loss: 0.05456123873591423
epoch: 2000 loss: 0.011234828270971775


In [10]:
# 3.4 모델 성능 평가

test_dataset = IrisDataset(train=False)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

num_correct = 0

with torch.no_grad():
  for data, target in test_dataloader:
    output = model(data)
    pred = torch.max(output, 1)[1]

    corr = pred.eq(target).sum().item()
    num_correct += corr

  print("Accuracy:", (num_correct/len(test_dataset.data))*100, "%")

Accuracy: 97.77777777777777 %


In [13]:
# p 122 까지 